# Week 1 Lab: Data Collection for Machine Learning

**CS 203: Software Tools and Techniques for AI**

---

## Lab Overview

In this lab, you will learn to collect data from the web using:

1. **HTTP fundamentals** - Understanding how the web works
2. **curl** - Command-line HTTP client
3. **Python requests** - Programmatic API calls
4. **BeautifulSoup** - Web scraping when APIs don't exist

**Goal**: Build a movie data collection pipeline for Netflix-style movie prediction.

---

## Setup

First, let's install and import the required libraries.

In [ ]:
# Install required packages (uncomment if needed)
# !pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
import time

print("All imports successful!")

All imports successful!


---

# Part 1: HTTP Fundamentals

Before we start collecting data, we need to understand how the web works.

## 1.1 Understanding URLs

A URL (Uniform Resource Locator) has several components:

```
https://api.omdbapi.com:443/v1/movies?t=Inception&y=2010#details
└─┬──┘ └──────┬───────┘└┬─┘└───┬───┘└─────────┬────────┘└───┬───┘
  │           │         │      │              │             │
Protocol    Host      Port   Path          Query        Fragment
```

### Question 1.1 (Solved): Parse a URL

Use Python's `urllib.parse` to break down a URL into its components.

In [ ]:
# SOLVED EXAMPLE
from urllib.parse import urlparse, parse_qs

url = "https://api.omdbapi.com/?apikey=demo&t=Inception&y=2010"

parsed = urlparse(url)

print(f"Scheme (protocol): {parsed.scheme}")
print(f"Host (domain): {parsed.netloc}")
print(f"Path: {parsed.path}")
print(f"Query string: {parsed.query}")

# Parse query parameters into a dictionary
params = parse_qs(parsed.query)
print(f"\nParsed parameters: {params}")

Scheme (protocol): https
Host (domain): api.omdbapi.com
Path: /
Query string: apikey=demo&t=Inception&y=2010

Parsed parameters: {'apikey': ['demo'], 't': ['Inception'], 'y': ['2010']}


### Question 1.2: Parse a Different URL

Parse the following GitHub API URL and extract:
1. The host
2. The path
3. All query parameters as a dictionary

URL: `https://api.github.com/search/repositories?q=machine+learning&sort=stars&order=desc`

In [ ]:
# YOUR CODE HERE
url = "https://api.github.com/search/repositories?q=machine+learning&sort=stars&order=desc"

# Parse the URL
parsed = urlparse(url)
# Print the host
print(f"Host (domain): {parsed.netloc}")
# Print the path
print(f"Path: {parsed.path}")
# Print the query parameters as a dictionary
params = parse_qs(parsed.query)
print(f"\nParsed parameters: {params}")

Host (domain): api.github.com
Path: /search/repositories

Parsed parameters: {'q': ['machine learning'], 'sort': ['stars'], 'order': ['desc']}


---

## 1.2 HTTP Status Codes

HTTP status codes tell you what happened with your request:

| Range | Category | Common Examples |
|-------|----------|----------------|
| 2xx | Success | 200 OK, 201 Created |
| 3xx | Redirect | 301 Moved, 302 Found |
| 4xx | Client Error | 400 Bad Request, 401 Unauthorized, 404 Not Found |
| 5xx | Server Error | 500 Internal Error, 503 Service Unavailable |

### Question 1.3: Match Status Codes

Match each scenario to the most likely HTTP status code:

1. You requested a movie that doesn't exist in the database
2. You made too many requests and hit the rate limit
3. Your API key is invalid
4. The request was successful and data was returned
5. The server crashed while processing your request

Status codes to choose from: `200`, `401`, `404`, `429`, `500`

In [ ]:
# YOUR ANSWERS HERE
answers = {
    "movie_not_found": 404,      # Replace None with the status code
    "rate_limited": 429,
    "invalid_api_key": 401,
    "success": 200,
    "server_crashed": 500
}

print(answers)

{'movie_not_found': 404, 'rate_limited': 429, 'invalid_api_key': 401, 'success': 200, 'server_crashed': 500}


---

# Part 2: Making Requests with `curl`

`curl` is a command-line tool for making HTTP requests. It's essential for quick testing.

## 2.1 Basic curl Commands

You can run shell commands in Jupyter using `!` prefix.

### Question 2.1 (Solved): Your First API Call

Let's call a simple public API that requires no authentication.

In [ ]:
# SOLVED EXAMPLE
# JSONPlaceholder is a free fake API for testing
!curl -s "https://jsonplaceholder.typicode.com/posts/1"

{
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}

### Question 2.2: Pretty Print with jq

The output above is hard to read. Use `jq` to format it nicely.

**Hint**: Pipe the curl output to jq: `curl ... | jq .`

In [ ]:
# YOUR CODE HERE
# Fetch the same post but format the output with jq
!curl -s "https://jsonplaceholder.typicode.com/posts/1" | jq

{
  "userId": 1,
  "id": 1,
  "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
  "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
}


### Question 2.3: Extract Specific Fields with jq

Fetch all posts from `https://jsonplaceholder.typicode.com/posts` and extract only the `title` field from each post.

**Hint**: Use `jq '.[].title'` to get the title from each element in the array.

In [ ]:
# YOUR CODE HERE
!curl -s "https://jsonplaceholder.typicode.com/posts" | jq '.[].id, .[].title'

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
"sunt aut facere repellat provident occaecati excepturi optio reprehenderit"
"qui est esse"
"ea molestias quasi exercitationem repellat qui ipsa sit aut"
"eum et est occaecati"
"nesciunt quas odio"
"dolorem eum magni eos aperiam quia"
"magnam facilis autem"
"dolorem dolore est ipsam"
"nesciunt iure omnis dolorem tempora et accusantium"
"optio molestias id quia eum"
"et ea vero quia laudantium autem"
"in quibusdam tempore odit est dolorem"
"dolorum ut in voluptas mollitia et saepe quo animi"
"voluptatem eligendi optio"
"eveniet quod temporibus"
"sint suscipit perspiciatis velit dolorum rerum ipsa laboriosam odio"
"fugit voluptas sed molestias voluptatem provident"
"voluptate et itaque vero tempora mo

### Question 2.4: View Response Headers

Use the `-I` flag to fetch only the response headers (no body) from:
`https://api.github.com`

What is the value of the `X-RateLimit-Limit` header?

In [ ]:
# YOUR CODE HERE
!curl -I "https://api.github.com" | grep -i x-ratelimit-limit

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0  2396    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
access-control-expose-headers: ETag, Link, Location, Retry-After, X-GitHub-OTP, X-RateLimit-Limit, X-RateLimit-Remaining, X-RateLimit-Used, X-RateLimit-Resource, X-RateLimit-Reset, X-OAuth-Scopes, X-Accepted-OAuth-Scopes, X-Poll-Interval, X-GitHub-Media-Type, X-GitHub-SSO, X-GitHub-Request-Id, Deprecation, Sunset
x-ratelimit-limit: 60


### Question 2.5: Add Custom Headers

Make a request to `https://httpbin.org/headers` with the following custom headers:
- `User-Agent: CS203-Lab/1.0`
- `Accept: application/json`

**Hint**: Use `-H "Header-Name: value"` for each header.

In [ ]:
# YOUR CODE HERE
!curl -H "User-Agent: CS203-Lab/1.0" "https://httpbin.org/headers"
!curl -H "Accept: application/json" "https://httpbin.org/headers"

{
  "headers": {
    "Accept": "*/*", 
    "Host": "httpbin.org", 
    "User-Agent": "CS203-Lab/1.0", 
    "X-Amzn-Trace-Id": "Root=1-69607e9a-1d4ad3d50c309e70279107ce"
  }
}
{
  "headers": {
    "Accept": "application/json", 
    "Host": "httpbin.org", 
    "User-Agent": "curl/7.81.0", 
    "X-Amzn-Trace-Id": "Root=1-69607e9a-71c069276ab307c0629c10a9"
  }
}


---

# Part 3: Python `requests` Library

While `curl` is great for testing, we need Python for automation.

## 3.1 Basic GET Requests

### Question 3.1 (Solved): Simple GET Request

Make a GET request and inspect the response object.

In [ ]:
# SOLVED EXAMPLE
import requests

response = requests.get("https://jsonplaceholder.typicode.com/posts/1")

print(f"Status Code: {response.status_code}")
print(f"Content-Type: {response.headers['Content-Type']}")
print(f"Response OK: {response.ok}")
print(f"\nJSON Data:")
print(response.json())

Status Code: 200
Content-Type: application/json; charset=utf-8
Response OK: True

JSON Data:
{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}


### Question 3.2: Fetch Multiple Posts

Fetch posts from `https://jsonplaceholder.typicode.com/posts` and:
1. Print the total number of posts
2. Print the titles of the first 5 posts

In [ ]:
# YOUR CODE HERE
count = 0;
response = requests.get("https://jsonplaceholder.typicode.com/posts")
for post in response.json():
  count+=1
print(count)
for i in range(5):
  print(f"{response.json()[i]['id']} : {response.json()[i]['title']}")

100
1 : sunt aut facere repellat provident occaecati excepturi optio reprehenderit
2 : qui est esse
3 : ea molestias quasi exercitationem repellat qui ipsa sit aut
4 : eum et est occaecati
5 : nesciunt quas odio


### Question 3.3 (Solved): Using Query Parameters

The proper way to add query parameters is using the `params` argument.

In [ ]:
# SOLVED EXAMPLE
import requests

# Bad way (manual string building)
# url = "https://jsonplaceholder.typicode.com/posts?userId=1"

# Good way (using params)
response = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"userId": 1,
            "id" : 1}
)
posts = response.json()
print(posts)
print(f"User 1 has {len(posts)} posts")
print(f"\nActual URL used: {response.url}")

[{'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}]
User 1 has 1 posts

Actual URL used: https://jsonplaceholder.typicode.com/posts?userId=1&id=1


### Question 3.4: Filter Posts by User

Fetch all posts by user 5 and user 7. Compare how many posts each user has.

**Hint**: Make two separate requests with different `userId` values.

In [ ]:
# YOUR CODE HERE
import requests
response = requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"userId": [5,7]}
)
posts = response.json()
print(posts)
print(f"User 5 and 7 has combined {len(posts)} posts")
print(f"\nActual URL used: {response.url}")

[{'userId': 5, 'id': 41, 'title': 'non est facere', 'body': 'molestias id nostrum\nexcepturi molestiae dolore omnis repellendus quaerat saepe\nconsectetur iste quaerat tenetur asperiores accusamus ex ut\nnam quidem est ducimus sunt debitis saepe'}, {'userId': 5, 'id': 42, 'title': 'commodi ullam sint et excepturi error explicabo praesentium voluptas', 'body': 'odio fugit voluptatum ducimus earum autem est incidunt voluptatem\nodit reiciendis aliquam sunt sequi nulla dolorem\nnon facere repellendus voluptates quia\nratione harum vitae ut'}, {'userId': 5, 'id': 43, 'title': 'eligendi iste nostrum consequuntur adipisci praesentium sit beatae perferendis', 'body': 'similique fugit est\nillum et dolorum harum et voluptate eaque quidem\nexercitationem quos nam commodi possimus cum odio nihil nulla\ndolorum exercitationem magnam ex et a et distinctio debitis'}, {'userId': 5, 'id': 44, 'title': 'optio dolor molestias sit', 'body': 'temporibus est consectetur dolore\net libero debitis vel velit

---

## 3.2 Working with Real APIs

Let's work with some real-world APIs.

### Question 3.5 (Solved): GitHub API - Public Repositories

The GitHub API is free to use (with rate limits) and doesn't require authentication for public data.

In [ ]:
# SOLVED EXAMPLE
import requests

# Fetch information about a popular repository
response = requests.get(
    "https://api.github.com/repos/pandas-dev/pandas",
    headers={"Accept": "application/vnd.github.v3+json"}
)

if response.ok:
    repo = response.json()
    print(f"Repository: {repo['full_name']}")
    print(f"Description: {repo['description']}")
    print(f"Stars: {repo['stargazers_count']:,}")
    print(f"Forks: {repo['forks_count']:,}")
    print(f"Language: {repo['language']}")
else:
    print(f"Error: {response.status_code}")

Repository: pandas-dev/pandas
Description: Flexible and powerful data analysis / manipulation library for Python, providing labeled data structures similar to R data.frame objects, statistical functions, and much more
Stars: 47,512
Forks: 19,485
Language: Python


### Question 3.6: Compare Popular ML Libraries

Fetch information about these ML-related repositories and create a comparison table:
- `scikit-learn/scikit-learn`
- `pytorch/pytorch`
- `tensorflow/tensorflow`

Show: name, stars, forks, and primary language.

**Hint**: Loop through the repos and collect data into a list of dictionaries, then create a DataFrame.

In [ ]:
# YOUR CODE HERE
repos = [
    "scikit-learn/scikit-learn",
    "pytorch/pytorch",
    "tensorflow/tensorflow"
]

# Fetch data for each repo
import requests
import pandas as pd
data = []
for repo in repos:
  url = f"https://api.github.com/repos/{repo}"
  response = requests.get(url).json()
  data.append({
      "name": response["name"],
      "stars": response["stargazers_count"],
      "forks": response["forks_count"],
      "language": response["language"]
  })
# Create a DataFrame
df = pd.DataFrame(data)
# Display the comparison
print(df)

           name   stars  forks language
0  scikit-learn   64569  26585   Python
1       pytorch   96473  26469   Python
2    tensorflow  193273  75144      C++


### Question 3.7: Search GitHub Repositories

Use the GitHub search API to find the top 10 most starred repositories with "machine learning" in their description.

API endpoint: `https://api.github.com/search/repositories`

Parameters:
- `q`: search query (e.g., "machine learning")
- `sort`: "stars"
- `order`: "desc"
- `per_page`: 10

Print the name and star count of each repository.

In [ ]:
# YOUR CODE HERE
import requests
import pandas as pd
data = []
url = "https://api.github.com/search/repositories"
response = requests.get(
    url,
    params={
        "q": "machine learning",
        "sort": "stars",
        "order": "desc",
        "per_page": 10
    }
)
print(response.status_code)
print(response.json())
for repo in response.json()["items"]:
  print(f"{repo['name']} : {repo['stargazers_count']}")


200
{'total_count': 986237, 'incomplete_results': False, 'items': [{'id': 45717250, 'node_id': 'MDEwOlJlcG9zaXRvcnk0NTcxNzI1MA==', 'name': 'tensorflow', 'full_name': 'tensorflow/tensorflow', 'private': False, 'owner': {'login': 'tensorflow', 'id': 15658638, 'node_id': 'MDEyOk9yZ2FuaXphdGlvbjE1NjU4NjM4', 'avatar_url': 'https://avatars.githubusercontent.com/u/15658638?v=4', 'gravatar_id': '', 'url': 'https://api.github.com/users/tensorflow', 'html_url': 'https://github.com/tensorflow', 'followers_url': 'https://api.github.com/users/tensorflow/followers', 'following_url': 'https://api.github.com/users/tensorflow/following{/other_user}', 'gists_url': 'https://api.github.com/users/tensorflow/gists{/gist_id}', 'starred_url': 'https://api.github.com/users/tensorflow/starred{/owner}{/repo}', 'subscriptions_url': 'https://api.github.com/users/tensorflow/subscriptions', 'organizations_url': 'https://api.github.com/users/tensorflow/orgs', 'repos_url': 'https://api.github.com/users/tensorflow/repo

---

## 3.3 Error Handling

Real-world APIs fail. We need to handle errors gracefully.

### Question 3.8 (Solved): Handling HTTP Errors

In [ ]:
# SOLVED EXAMPLE
import requests

def fetch_with_error_handling(url):
    """Fetch URL with proper error handling."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raises exception for 4xx/5xx
        return response.json()
    except requests.exceptions.Timeout:
        print(f"Timeout: Request took too long")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e.response.status_code}")
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
    return None

# Test with valid URL
print("Valid URL:")
data = fetch_with_error_handling("https://jsonplaceholder.typicode.com/posts/1")
if data:
    print(f"  Got post: {data['title'][:50]}...")

# Test with invalid URL (404)
print("\nInvalid URL (404):")
fetch_with_error_handling("https://jsonplaceholder.typicode.com/posts/99999")

Valid URL:
  Got post: sunt aut facere repellat provident occaecati excep...

Invalid URL (404):
HTTP Error: 404


### Question 3.9: Robust Fetcher Function

Write a function `safe_fetch(url, max_retries=3)` that:

1. Attempts to fetch the URL
2. If it fails with a 5xx error, retries up to `max_retries` times
3. Waits 1 second between retries
4. Returns the JSON data if successful, None otherwise

Test it with `https://httpbin.org/status/500` (always returns 500) and `https://jsonplaceholder.typicode.com/posts/1` (always works).

In [ ]:
# YOUR CODE HERE
import time

def safe_fetch(url, max_retries=3):
    """Fetch URL with retry logic for server errors."""
    for i in range(max_retries):
      try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json()
      except requests.exceptions.Timeout:
        print(f"Timeout: Request took too long")
        time.sleep(1)
      except requests.exceptions.HTTPError as e:
        print(f"HTTP Error: {e.response.status_code}")
        if e.response.status_code == 500:
          time.sleep(1)
        else:
          return None

# Test your function
print("Testing with working URL:")
result = safe_fetch("https://jsonplaceholder.typicode.com/posts/1")
print(f"Result: {result}")

print("\nTesting with failing URL (500):")
result = safe_fetch("https://httpbin.org/status/500")
print(f"Result: {result}")

Testing with working URL:
Result: {'userId': 1, 'id': 1, 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit', 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

Testing with failing URL (500):
HTTP Error: 500
HTTP Error: 500
HTTP Error: 500
Result: None


---

# Part 4: The OMDb Movie API

Now let's work with the OMDb API - our main data source for the Netflix project.

**Note**: You need an API key from https://www.omdbapi.com/apikey.aspx (free tier available).

For this lab, we'll use a demo key that has limited functionality.

In [ ]:
# Set your API key here
# Get a free key from: https://www.omdbapi.com/apikey.aspx
# OMDB_API_KEY = "http://www.omdbapi.com/?i=tt3896198&apikey=a6a8c3a6"  # Replace with your actual key

# For demo purposes, you can try with key "demo" but it's very limited
OMDB_API_KEY = "demo"

### Question 4.1 (Solved): Fetch a Single Movie

In [ ]:
# SOLVED EXAMPLE
import requests
OMDB_API_KEY = "a6a8c3a6"
def fetch_movie(title, year=None, api_key=OMDB_API_KEY):
    """Fetch movie data from OMDb API."""
    params = {
        "apikey": api_key,
        "t": title,  # Search by title
        "type": "movie"
    }
    if year:
        params["y"] = year

    response = requests.get("https://www.omdbapi.com/", params=params)

    if response.ok:
        data = response.json()
        if data.get("Response") == "True":
            return data
        else:
            print(f"Movie not found: {data.get('Error')}")
    return None

# Fetch Inception
# movie = fetch_movie("Inception")
movie = fetch_movie("Mr. Nobody")
if movie:
    print(f"Title: {movie['Title']}")
    print(f"Year: {movie['Year']}")
    print(f"Director: {movie['Director']}")
    print(f"IMDB Rating: {movie['imdbRating']}")
    print(f"Genre: {movie['Genre']}")
else :
  print("Failed")

Title: Mr. Nobody
Year: 2009
Director: Jaco Van Dormael
IMDB Rating: 7.7
Genre: Drama, Fantasy, Romance


### Question 4.2: Explore the Response

Fetch data for "The Dark Knight" and print ALL available fields in the response.

Which fields might be useful for predicting movie success?

In [ ]:
# YOUR CODE HERE
movie = fetch_movie("The Dark Knight")
movie_usefull_features = ["Genre","Director","Actors","Writer","Plot","Language","Awards","imdbRating","BoxOffice"]
if movie:
    print("Following are the Usefull Movie Success features ->")
    for key, value in movie.items():
      if(key in movie_usefull_features):
        print(f"{key}: {value}")
      # print(f"{key}: {value}")
    # print(f"Title: {movie['Title']}")
    # print(f"Year: {movie['Year']}")
    # print(f"Director: {movie['Director']}")
    # print(f"IMDB Rating: {movie['imdbRating']}")
    # print(f"Genre: {movie['Genre']}")
else :
  print("Failed")

Following are the Usefull Movie Success features ->
Genre: Action, Crime, Drama
Director: Christopher Nolan
Writer: Jonathan Nolan, Christopher Nolan, David S. Goyer
Actors: Christian Bale, Heath Ledger, Aaron Eckhart
Plot: When a menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman, James Gordon and Harvey Dent must work together to put an end to the madness.
Language: English, Mandarin
Awards: Won 2 Oscars. 163 wins & 165 nominations total
imdbRating: 9.1
BoxOffice: $534,987,076


### Question 4.3: Fetch Multiple Movies

Create a function `fetch_movies(titles)` that:
1. Takes a list of movie titles
2. Fetches data for each movie
3. Returns a list of movie dictionaries (only successful fetches)
4. Adds a 0.5 second delay between requests (to respect rate limits)

Test it with: `["Inception", "The Matrix", "Interstellar", "NonExistentMovie123"]`

In [ ]:
# YOUR CODE HERE
import requests
OMDB_API_KEY = "a6a8c3a6"
def fetch_movie(title, year=None, api_key=OMDB_API_KEY):
    """Fetch movie data from OMDb API."""
    params = {
        "apikey": api_key,
        "t": title,  # Search by title
        "type": "movie"
    }
    if year:
        params["y"] = year

    response = requests.get("https://www.omdbapi.com/", params=params)

    if response.ok:
        data = response.json()
        if data.get("Response") == "True":
            return data
    return None
movie_dict = {}
def fetch_movies(titles):

    """Fetch multiple movies from OMDb API."""
    for movie in titles:
      movie_rcvd = fetch_movie(movie)
      if movie_rcvd:
        movie_dict[movie] = movie_rcvd
      time.sleep(0.5)

# Test
test_titles = ["Inception", "The Matrix", "Interstellar", "NonExistentMovie123"]
movies = fetch_movies(test_titles)
print(f"Successfully fetched {len(movie_dict)} out of {len(test_titles)} movies")

Successfully fetched 3 out of 4 movies


### Question 4.4: Create a Movie DataFrame

Using the movies you fetched, create a pandas DataFrame with these columns:
- title
- year (as integer)
- genre
- director
- imdb_rating (as float)
- imdb_votes (as integer, remove commas)
- runtime_minutes (as integer, extract from "148 min")
- box_office (keep as string for now)

**Hint**: You'll need to clean the data types.

In [ ]:
# YOUR CODE HERE
data = []
# for movie in movie_dict:
#   data.append(movie_dict[movie])

# df = pd.DataFrame(data)
# df
for movie in movie_dict.values():
    data.append({
        "title": movie.get("Title"),
        "year": int(movie["Year"]) if movie.get("Year", "").isdigit() else None,
        "genre": movie.get("Genre"),
        "director": movie.get("Director"),
        "imdb_rating": float(movie["imdbRating"]) if movie.get("imdbRating") != "N/A" else None,
        "imdb_votes": int(movie["imdbVotes"].replace(",", "")) if movie.get("imdbVotes") != "N/A" else None,
        "runtime_minutes": int(movie["Runtime"].split()[0]) if movie.get("Runtime") != "N/A" else None,
        "box_office": movie.get("BoxOffice")
    })
df = pd.DataFrame(data)
df

,title,year,genre,director,imdb_rating,imdb_votes,runtime_minutes,box_office
0,Inception,2010,"Action, Adventure, Sci-Fi",Christopher Nolan,8.8,2767518,148,"$292,587,330"
1,The Matrix,1999,"Action, Sci-Fi","Lana Wachowski, Lilly Wachowski",8.7,2217731,136,"$177,559,005"
2,Interstellar,2014,"Adventure, Drama, Sci-Fi",Christopher Nolan,8.7,2454660,169,"$203,227,580"


### Question 4.5: Search Movies by Title

OMDb also has a search endpoint that returns multiple results.

Use the `s` parameter instead of `t` to search for movies containing "Star Wars".

API endpoint: `https://www.omdbapi.com/?apikey=YOUR_KEY&s=Star Wars&type=movie`

Print the title and year of each result.

In [ ]:
# YOUR CODE HERE
def fetch_movie_by_s(title, year=None, api_key=OMDB_API_KEY):
    """Fetch movie data from OMDb API."""
    params = {
        "apikey": api_key,
        "s": title,
        "type": "movie"
    }
    if year:
        params["y"] = year

    response = requests.get("https://www.omdbapi.com/", params=params)

    if response.ok:
        data = response.json()
        if data.get("Response") == "True":
            return data
    return None
movie = fetch_movie_by_s("Star Wars", api_key=OMDB_API_KEY)
data = []
for m in movie['Search']:
  data.append({
      "title": m.get("Title"),
      "year": int(m["Year"])
  })
df = pd.DataFrame(data)
df

,title,year
0,Star Wars: Episode IV - A New Hope,1977
1,Star Wars: Episode V - The Empire Strikes Back,1980
2,Star Wars: Episode VI - Return of the Jedi,1983
3,Star Wars: Episode VII - The Force Awakens,2015
4,Star Wars: Episode I - The Phantom Menace,1999
5,Star Wars: Episode III - Revenge of the Sith,2005
6,Star Wars: Episode II - Attack of the Clones,2002
7,Rogue One: A Star Wars Story,2016
8,Star Wars: Episode VIII - The Last Jedi,2017
9,Star Wars: Episode IX - The Rise of Skywalker,2019


### Question 4.6: Handle Pagination

The OMDb search API returns 10 results per page and includes a `totalResults` field.

Write a function `search_all_movies(query)` that:
1. Searches for movies matching the query
2. Fetches ALL pages of results (use the `page` parameter)
3. Returns a list of all movies found

**Hint**: `totalResults` tells you how many movies exist. Divide by 10 to get the number of pages.

Test with a query that has many results like "Batman".

In [ ]:
# YOUR CODE HERE
def search_all_movies(query, api_key=OMDB_API_KEY):
    """Search OMDb and return ALL matching movies across all pages."""
    movies = fetch_movie_by_s(query, api_key=api_key)
    pages = int(movies['totalResults']) // 10
    data = []
    for i in range(1,pages+1):
      params={
                "apikey": api_key,
                "s": query,
                "type": "movie",
                "page": i
            }
      response = requests.get("https://www.omdbapi.com/", params=params)
      movie = response.json()
      if movie.get("Response") == "True":
            for m in movie.get("Search", []):
                data.append(m.get("Title"))
      else:
            break
      for m in movie['Search']:
        data.append(m.get("Title"))

    return data

# Test
all_batman = search_all_movies("Batman")
print(f"Found {len(all_batman)} Batman movies")
df = pd.DataFrame(all_batman)
df

Found 1020 Batman movies


,0
0,Batman Begins
1,The Batman
2,Batman v Superman: Dawn of Justice
3,Batman v Superman: Dawn of Justice
4,Batman
...,...
1015,Batman: Chaos in Gotham City
1016,Batman: The Enemy Within
1017,Batman Consquences
1018,Batman Consequences: Ending with the Premiere


---

# Part 5: Web Scraping with BeautifulSoup

When APIs don't exist or don't have what we need, we scrape.

## 5.1 HTML Basics

### Question 5.1 (Solved): Parse HTML

In [ ]:
# SOLVED EXAMPLE
from bs4 import BeautifulSoup

html = """
<html>
<body>
    <div class="movie" id="movie-1">
        <h2 class="title">Inception</h2>
        <span class="year">2010</span>
        <span class="rating">8.8</span>
        <a href="/movies/inception">More Info</a>
    </div>
    <div class="movie" id="movie-2">
        <h2 class="title">The Matrix</h2>
        <span class="year">1999</span>
        <span class="rating">8.7</span>
        <a href="/movies/matrix">More Info</a>
    </div>
</body>
</html>
"""

soup = BeautifulSoup(html, 'html.parser')

# Find all movie divs
movies = soup.find_all('div', class_='movie')
print(f"Found {len(movies)} movies\n")

# Extract data from each
for movie in movies:
    title = movie.find('h2', class_='title').text
    year = movie.find('span', class_='year').text
    rating = movie.find('span', class_='rating').text
    link = movie.find('a')['href']

    print(f"{title} ({year}) - Rating: {rating} - Link: {link}")

Found 2 movies

Inception (2010) - Rating: 8.8 - Link: /movies/inception
The Matrix (1999) - Rating: 8.7 - Link: /movies/matrix


### Question 5.2: CSS Selectors

Rewrite the above extraction using CSS selectors (`.select()` and `.select_one()`) instead of `.find()` and `.find_all()`.

**Hint**:
- `.movie` selects elements with class "movie"
- `.movie .title` selects elements with class "title" inside class "movie"

In [ ]:
# YOUR CODE HERE
# Use the same 'soup' from above

# Extract using CSS selectors
movies = soup.select('.movie')
print(f"Found {len(movies)} movies\n")
for movie in movies:
  title = movie.select_one('.title').text
  year = movie.select_one('.year').text
  rating = movie.select_one('.rating').text
  link = movie.select_one('a')['href']
  print(f"{title} ({year}) - Rating: {rating} - Link: {link}")

Found 2 movies

Inception (2010) - Rating: 8.8 - Link: /movies/inception
The Matrix (1999) - Rating: 8.7 - Link: /movies/matrix


### Question 5.3: Scrape a Real Website

Let's scrape the example website `http://quotes.toscrape.com/` which is designed for scraping practice.

Extract all quotes from the first page, including:
- The quote text
- The author name
- The tags

Return the results as a list of dictionaries.

In [ ]:
# YOUR CODE HERE
import requests
from bs4 import BeautifulSoup as BFS

# Fetch the page
url = "http://quotes.toscrape.com/"
site = requests.get(url)
# print(site.text)
# Parse the HTML
soup = BFS(requests.get(url).text, "html.parser")
# # Extract quotes
quotes = []
for quote in soup.select(".quote"):
  quotes.append(quote.select_one(".text").text)

# Print results
for val,i in enumerate(quotes):
  print(val,i)

0 “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
1 “It is our choices, Harry, that show what we truly are, far more than our abilities.”
2 “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
3 “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
4 “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
5 “Try not to become a man of success. Rather become a man of value.”
6 “It is better to be hated for what you are than to be loved for what you are not.”
7 “I have not failed. I've just found 10,000 ways that won't work.”
8 “A woman is like a tea bag; you never know how strong it is until it's in hot water.”
9 “A day without sunshine is like, you know, night.”


### Question 5.4: Handle Pagination in Scraping

The quotes website has multiple pages. Scrape the first 3 pages and collect all quotes.

Pages follow the pattern:
- Page 1: `http://quotes.toscrape.com/page/1/`
- Page 2: `http://quotes.toscrape.com/page/2/`
- etc.

**Remember**: Add a delay between requests to be polite!

In [ ]:
# YOUR CODE HERE
quotes = []
for i in range (3):
  url = f"http://quotes.toscrape.com/page/{i+1}/"
  site = requests.get(url)
  soup = BFS(requests.get(url).text, "html.parser")
  for quote in soup.select(".quote"):
    quotes.append(quote.select_one(".text").text)
  time.sleep(1)
for i in range(len(quotes)):
  print(i,":",quotes[i])

0 : “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
1 : “It is our choices, Harry, that show what we truly are, far more than our abilities.”
2 : “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”
3 : “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”
4 : “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”
5 : “Try not to become a man of success. Rather become a man of value.”
6 : “It is better to be hated for what you are than to be loved for what you are not.”
7 : “I have not failed. I've just found 10,000 ways that won't work.”
8 : “A woman is like a tea bag; you never know how strong it is until it's in hot water.”
9 : “A day without sunshine is like, you know, night.”
10 : “This life is what you make it. No matter what, yo

### Question 5.5: Extract Table Data

Scrape the table from `https://www.w3schools.com/html/html_tables.asp`.

The table contains company data. Extract all rows and create a pandas DataFrame.

**Hint**: Look for `<table>`, `<tr>` (table row), `<th>` (header), and `<td>` (data cell) elements.

In [ ]:
# YOUR CODE HERE
# Hint: pandas has a read_html() function that can do this automatically!
# But try doing it manually first to understand the process.
import requests
import pandas as pd
from bs4 import BeautifulSoup as BFS
url = "https://www.w3schools.com/html/html_tables.asp"
site = requests.get(url)
soup = BFS(requests.get(url).text, "html.parser")
# site.text
table = soup.select_one("table")
rows = []
temp = []
for cell_data in table.select("th"):
  temp.append(cell_data)
rows.append(temp)
for row in table.select("tr"):
  cells = []
  for cell in row.select("td"):
    cells.append(cell.text)
  rows.append(cells)
df = pd.DataFrame(rows)
df

,0,1,2
0,[Company],[Contact],[Country]
1,None,None,None
2,Alfreds Futterkiste,Maria Anders,Germany
3,Centro comercial Moctezuma,Francisco Chang,Mexico
4,Ernst Handel,Roland Mendel,Austria
5,Island Trading,Helen Bennett,UK
6,Laughing Bacchus Winecellars,Yoshi Tannamuri,Canada
7,Magazzini Alimentari Riuniti,Giovanni Rovelli,Italy


---

# Part 6: Building the Movie Data Pipeline

Now let's put everything together to build a complete data collection pipeline for our Netflix project.

## 6.1 The Complete Pipeline

### Question 6.1 (Solved): Movie Data Collector Class

In [ ]:
# SOLVED EXAMPLE
import requests
import pandas as pd
import time
from typing import List, Dict, Optional

class MovieDataCollector:
    """Collect movie data from OMDb API."""

    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "http://www.omdbapi.com/"
        self.delay = 0.5  # Seconds between requests

    def fetch_movie(self, title: str, year: Optional[int] = None) -> Optional[Dict]:
        """Fetch a single movie by title."""
        params = {
            "apikey": self.api_key,
            "t": title,
            "type": "movie"
        }
        if year:
            params["y"] = year

        try:
            response = requests.get(self.base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            if data.get("Response") == "True":
                return data
        except Exception as e:
            print(f"Error fetching {title}: {e}")

        return None

    def fetch_movies(self, titles: List[str]) -> List[Dict]:
        """Fetch multiple movies."""
        movies = []

        for i, title in enumerate(titles):
            print(f"Fetching {i+1}/{len(titles)}: {title}")
            movie = self.fetch_movie(title)

            if movie:
                movies.append(movie)

            time.sleep(self.delay)

        return movies
    def fetch_movie_by_id(self, imdb_id: str) -> Optional[Dict]:
        """Fetch full movie details using IMDb ID."""
        params = {
            "apikey": self.api_key,
            "i": imdb_id,
            "type": "movie"
        }

        try:
            response = requests.get(self.base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            if data.get("Response") == "True":
                return data
        except Exception as e:
            print(f"Error fetching {imdb_id}: {e}")

        return None

    # -----------------------------
    # FILTER BY GENRE (LOCAL)
    # -----------------------------
    def filter_by_genre(self, movies: List[Dict], genres: List[str]) -> List[Dict]:
        """Filter movies by genre (case-insensitive)."""
        genres = [g.lower() for g in genres]
        filtered = []

        for m in movies:
            movie_genres = m.get("Genre", "").lower()
            if any(g in movie_genres for g in genres):
                filtered.append(m)

        return filtered

    def to_dataframe(self, movies: List[Dict]) -> pd.DataFrame:
        """Convert movie data to cleaned DataFrame."""
        if not movies:
            return pd.DataFrame()

        # Extract relevant fields
        rows = []
        for m in movies:
            rows.append({
                "title": m.get("Title"),
                "year": m.get("Year"),
                "genre": m.get("Genre"),
                "director": m.get("Director"),
                "actors": m.get("Actors"),
                "imdb_rating": m.get("imdbRating"),
                "imdb_votes": m.get("imdbVotes"),
                "runtime": m.get("Runtime"),
                "box_office": m.get("BoxOffice"),
                "imdb_id": m.get("imdbID")
            })

        df = pd.DataFrame(rows)

        # Clean data types
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["imdb_rating"] = pd.to_numeric(df["imdb_rating"], errors="coerce")
        df["imdb_votes"] = df["imdb_votes"].str.replace(",", "").pipe(pd.to_numeric, errors="coerce").astype("Int64")
        # Fix: str.extract returns a DataFrame, we need column 0 to get a Series
        df["runtime_min"] = df["runtime"].str.extract(r"(\d+)")[0].pipe(pd.to_numeric, errors="coerce").astype("Int64")

        return df

# Usage example
OMDB_API_KEY = "a6a8c3a6"
collector = MovieDataCollector(OMDB_API_KEY)
movies = collector.fetch_movies(["Inception", "The Matrix"])
df = collector.to_dataframe(movies)
df

Fetching 1/2: Inception
Fetching 2/2: The Matrix


,title,year,genre,director,actors,imdb_rating,imdb_votes,runtime,box_office,imdb_id,runtime_min
0,Inception,2010,"Action, Adventure, Sci-Fi",Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ellio...",8.8,2767518,148 min,"$292,587,330",tt1375666,148
1,The Matrix,1999,"Action, Sci-Fi","Lana Wachowski, Lilly Wachowski","Keanu Reeves, Laurence Fishburne, Carrie-Anne ...",8.7,2217731,136 min,"$177,559,005",tt0133093,136


### Question 6.2: Add Search Functionality

Extend the `MovieDataCollector` class to add a `search_movies(query, max_results=50)` method that:
1. Searches for movies matching the query
2. Handles pagination to get up to `max_results` movies
3. For each search result, fetches the full movie details
4. Returns the detailed movie data

**Hint**: Search results only contain basic info (title, year, poster, imdbID). You need to use the imdbID to fetch full details.

In [ ]:
from re import search
# YOUR CODE HERE
# Extend the MovieDataCollector class or add a method
# SOLVED EXAMPLE
import requests
import pandas as pd
import time
from typing import List, Dict, Optional
OMDB_API_KEY = "a6a8c3a6"
class MovieDataCollector:
    """Collect movie data from OMDb API."""

    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "http://www.omdbapi.com/"
        self.delay = 0.5  # Seconds between requests

    # def fetch_movie(self, title: str, year: Optional[int] = None) -> Optional[Dict]:
    #     """Fetch a single movie by title."""
    #     params = {
    #         "apikey": self.api_key,
    #         "t": title,
    #         "type": "movie"
    #     }
    #     if year:
    #         params["y"] = year

    #     try:
    #         response = requests.get(self.base_url, params=params, timeout=10)
    #         response.raise_for_status()
    #         data = response.json()

    #         if data.get("Response") == "True":
    #             return data
    #     except Exception as e:
    #         print(f"Error fetching {title}: {e}")

    #     return None
    def fetch_movie_by_s(self, title, year=None, api_key=OMDB_API_KEY):
        params = {
            "apikey": self.api_key,
            "s": title,
            "type": "movie"
        }
        if year:
            params["y"] = year
        try:
            response = requests.get(self.base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            if data.get("Response") == "True":
                return data
        except Exception as e:
            print(f"Error fetching {title}: {e}")
        return None

    def search_movies(self, query: str,api_key=OMDB_API_KEY, max_results: int = 50) -> List[Dict]:
        all_movies = []
        movies = self.fetch_movie_by_s(query)
        pages = int(movies['totalResults']) // 10
        data = []
        for i in range(1,pages+1):
          params={
                    "apikey": api_key,
                    "s": query,
                    "type": "movie",
                    "page": i
                }
          response = requests.get("https://www.omdbapi.com/", params=params)
          movie = response.json()

          if movie.get("Response") == "True":
                for m in movie.get("Search", []):
                    data.append(m.get("Title"))
          else:
                break
          # for m in movie['Search']:
          #   data.append(m.get("Title"))

        return data


    # def fetch_movies(self, titles: List[str]) -> List[Dict]:
    #     """Fetch multiple movies."""
    #     movies = []

    #     for i, title in enumerate(titles):
    #         print(f"Fetching {i+1}/{len(titles)}: {title}")
    #         movie = self.fetch_movie(title)

    #         if movie:
    #             movies.append(movie)

    #         time.sleep(self.delay)

    #     return movies

    def to_dataframe(self, movies: List[Dict]) -> pd.DataFrame:
        """Convert movie data to cleaned DataFrame."""
        if not movies:
            return pd.DataFrame()

        # Extract relevant fields
        rows = []
        for m in movies:
            rows.append({
                "title": m.get("Title"),
                "year": m.get("Year"),
                "genre": m.get("Genre"),
                "director": m.get("Director"),
                "actors": m.get("Actors"),
                "imdb_rating": m.get("imdbRating"),
                "imdb_votes": m.get("imdbVotes"),
                "runtime": m.get("Runtime"),
                "box_office": m.get("BoxOffice"),
                "imdb_id": m.get("imdbID")
            })

        df = pd.DataFrame(rows)

        # Clean data types
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["imdb_rating"] = pd.to_numeric(df["imdb_rating"], errors="coerce")
        df["imdb_votes"] = df["imdb_votes"].str.replace(",", "").pipe(pd.to_numeric, errors="coerce").astype("Int64")
        # Fix: str.extract returns a DataFrame, we need column 0 to get a Series
        df["runtime_min"] = df["runtime"].str.extract(r"(\d+)")[0].pipe(pd.to_numeric, errors="coerce").astype("Int64")

        return df

# Usage example

collector = MovieDataCollector(OMDB_API_KEY)
# movies = collector.fetch_movies(["Inception", "The Matrix"])
all_batman = collector.search_movies("batman")
print(f"Found {len(all_batman)} Batman movies")
df = pd.DataFrame(all_batman)
df

Found 510 Batman movies


,0
0,Batman Begins
1,The Batman
2,Batman v Superman: Dawn of Justice
3,Batman v Superman: Dawn of Justice
4,Batman
...,...
505,Batman: Chaos in Gotham City
506,Batman: The Enemy Within
507,Batman Consquences
508,Batman Consequences: Ending with the Premiere


### Question 6.3: Build a Genre-Based Dataset

Use your collector to build a dataset of popular movies from different genres:

1. Search for 10 movies each for: "action", "comedy", "drama", "horror", "sci-fi"
2. Combine all results into a single DataFrame
3. Remove any duplicates (some movies might appear in multiple searches)
4. Save to CSV

**Note**: This might take a while due to rate limiting. Start with fewer movies for testing.

In [ ]:
class MovieDataCollector:
    """Collect movie data from OMDb API."""

    def __init__(self, api_key: str):
        self.api_key = api_key
        self.base_url = "http://www.omdbapi.com/"
        self.delay = 0.5  # Seconds between requests
    def fetch_movie_by_s(self, title, year=None, api_key=OMDB_API_KEY):
        params = {
            "apikey": self.api_key,
            "s": title,
            "type": "movie"
        }
        if year:
            params["y"] = year
        try:
            response = requests.get(self.base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()
            if data.get("Response") == "True":
                return data
        except Exception as e:
            print(f"Error fetching {title}: {e}")
        return None

    def search_movies(self, query: str,api_key=OMDB_API_KEY, max_results: int = 50) -> List[Dict]:
        all_movies = []
        movies = self.fetch_movie_by_s(query)
        pages = int(movies['totalResults']) // 10
        data = []
        for i in range(1,pages+1):
          params={
                    "apikey": api_key,
                    "s": query,
                    "type": "movie",
                    "page": i
                }
          response = requests.get("https://www.omdbapi.com/", params=params)
          movie = response.json()

          if movie.get("Response") == "True":
                for m in movie.get("Search", []):
                    data.append(m.get("Title"))
          else:
                break
          # for m in movie['Search']:
          #   data.append(m.get("Title"))

        return data
    def fetch_movie(self, title: str, year: Optional[int] = None) -> Optional[Dict]:
        """Fetch a single movie by title."""
        params = {
            "apikey": self.api_key,
            "t": title,
            "type": "movie"
        }
        if year:
            params["y"] = year

        try:
            response = requests.get(self.base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            if data.get("Response") == "True":
                return data
        except Exception as e:
            print(f"Error fetching {title}: {e}")

        return None

    def fetch_movies(self, titles: List[str]) -> List[Dict]:
        """Fetch multiple movies."""
        movies = []

        for i, title in enumerate(titles):
            print(f"Fetching {i+1}/{len(titles)}: {title}")
            movie = self.fetch_movie(title)

            if movie:
                movies.append(movie)

            time.sleep(self.delay)

        return movies
    def fetch_movie_by_id(self, imdb_id: str) -> Optional[Dict]:
        """Fetch full movie details using IMDb ID."""
        params = {
            "apikey": self.api_key,
            "i": imdb_id,
            "type": "movie"
        }

        try:
            response = requests.get(self.base_url, params=params, timeout=10)
            response.raise_for_status()
            data = response.json()

            if data.get("Response") == "True":
                return data
        except Exception as e:
            print(f"Error fetching {imdb_id}: {e}")

        return None

    # -----------------------------
    # FILTER BY GENRE (LOCAL)
    # -----------------------------
    def filter_by_genre(self, movies: List[Dict], genres: List[str]) -> List[Dict]:
        """Filter movies by genre (case-insensitive)."""
        genres = [g.lower() for g in genres]
        filtered = []

        for m in movies:
            movie_genres = m.get("Genre", "").lower()
            if any(g in movie_genres for g in genres):
                filtered.append(m)

        return filtered

    def to_dataframe(self, movies: List[Dict]) -> pd.DataFrame:
        """Convert movie data to cleaned DataFrame."""
        if not movies:
            return pd.DataFrame()

        # Extract relevant fields
        rows = []
        for m in movies:
            rows.append({
                "title": m.get("Title"),
                "year": m.get("Year"),
                "genre": m.get("Genre"),
                "director": m.get("Director"),
                "actors": m.get("Actors"),
                "imdb_rating": m.get("imdbRating"),
                "imdb_votes": m.get("imdbVotes"),
                "runtime": m.get("Runtime"),
                "box_office": m.get("BoxOffice"),
                "imdb_id": m.get("imdbID")
            })

        df = pd.DataFrame(rows)

        # Clean data types
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["imdb_rating"] = pd.to_numeric(df["imdb_rating"], errors="coerce")
        df["imdb_votes"] = df["imdb_votes"].str.replace(",", "").pipe(pd.to_numeric, errors="coerce").astype("Int64")
        # Fix: str.extract returns a DataFrame, we need column 0 to get a Series
        df["runtime_min"] = df["runtime"].str.extract(r"(\d+)")[0].pipe(pd.to_numeric, errors="coerce").astype("Int64")

        return df

In [ ]:
import requests
import pandas as pd
import time
from typing import List, Dict, Optional

# -----------------------------
# CONFIG
# -----------------------------
OMDB_API_KEY = "a6a8c3a6"
BASE_URL = "https://www.omdbapi.com/"
RATE_LIMIT_DELAY = 0.4  # seconds


# -----------------------------
# MOVIE DATA COLLECTOR
# -----------------------------
class MovieDataCollector:
    def __init__(self, api_key: str):
        self.api_key = api_key

    # Search movies by keyword (s=)
    def search_movies(self, query: str, max_results: int = 10) -> List[Dict]:
        results = []

        params = {
            "apikey": self.api_key,
            "s": query,
            "type": "movie",
            "page": 1
        }

        response = requests.get(BASE_URL, params=params, timeout=10)
        data = response.json()

        if data.get("Response") != "True":
            return []

        total_results = int(data["totalResults"])
        total_pages = min((total_results + 9) // 10, (max_results + 9) // 10)

        results.extend(data.get("Search", []))

        for page in range(2, total_pages + 1):
            params["page"] = page
            response = requests.get(BASE_URL, params=params, timeout=10)
            data = response.json()

            if data.get("Response") != "True":
                break

            results.extend(data.get("Search", []))
            time.sleep(RATE_LIMIT_DELAY)

        return results[:max_results]

    # Fetch full movie details (i=)
    def fetch_movie_by_id(self, imdb_id: str) -> Optional[Dict]:
        params = {
            "apikey": self.api_key,
            "i": imdb_id,
            "type": "movie"
        }

        try:
            response = requests.get(BASE_URL, params=params, timeout=10)
            data = response.json()
            if data.get("Response") == "True":
                return data
        except Exception as e:
            print(f"Error fetching {imdb_id}: {e}")

        return None

    # Convert to DataFrame
    def to_dataframe(self, movies: List[Dict]) -> pd.DataFrame:
        rows = []
        for m in movies:
            rows.append({
                "title": m.get("Title"),
                "year": m.get("Year"),
                "genre": m.get("Genre"),
                "director": m.get("Director"),
                "actors": m.get("Actors"),
                "imdb_rating": m.get("imdbRating"),
                "imdb_votes": m.get("imdbVotes"),
                "runtime": m.get("Runtime"),
                "imdb_id": m.get("imdbID")
            })

        df = pd.DataFrame(rows)

        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        df["imdb_rating"] = pd.to_numeric(df["imdb_rating"], errors="coerce")
        df["imdb_votes"] = (
            df["imdb_votes"]
            .str.replace(",", "", regex=False)
            .pipe(pd.to_numeric, errors="coerce")
        )
        df["runtime_min"] = (
            df["runtime"]
            .str.extract(r"(\d+)")[0]
            .pipe(pd.to_numeric, errors="coerce")
        )

        return df


In [ ]:
collector = MovieDataCollector(OMDB_API_KEY)

# Genre → search keywords (important workaround)
genre_queries = {
    "action": ["action", "battle", "war"],
    "comedy": ["comedy", "funny", "party"],
    "drama": ["drama", "life", "family"],
    "horror": ["horror", "ghost", "haunted"],
    "sci-fi": ["sci-fi", "space", "future"]
}

MOVIES_PER_GENRE = 10

all_movies = []

# -----------------------------
# SEARCH + FETCH
# -----------------------------
for genre, queries in genre_queries.items():
    print(f"Collecting {genre} movies...")
    collected = []

    for q in queries:
        if len(collected) >= MOVIES_PER_GENRE:
            break

        search_results = collector.search_movies(q, max_results=10)

        for m in search_results:
            if len(collected) >= MOVIES_PER_GENRE:
                break

            movie = collector.fetch_movie_by_id(m["imdbID"])
            if movie and genre.lower() in movie.get("Genre", "").lower():
                collected.append(movie)
                time.sleep(RATE_LIMIT_DELAY)

    all_movies.extend(collected)

# -----------------------------
# DEDUPLICATE
# -----------------------------
unique_movies = {}
for m in all_movies:
    unique_movies[m["imdbID"]] = m

final_movies = list(unique_movies.values())

print(f"\nTotal unique movies collected: {len(final_movies)}")

# -----------------------------
# DATAFRAME + CSV
# -----------------------------
df = collector.to_dataframe(final_movies)
df.to_csv("genre_based_movies.csv", index=False)

df.head()



Total unique movies collected: 50


,title,year,genre,director,actors,imdb_rating,imdb_votes,runtime,imdb_id,runtime_min
0,Last Action Hero,1993,"Action, Adventure, Comedy",John McTiernan,"Arnold Schwarzenegger, F. Murray Abraham, Art ...",6.5,171378,130 min,tt0107362,130.0
1,Back in Action,2025,"Action, Comedy",Seth Gordon,"Jamie Foxx, Cameron Diaz, McKenna Roberts",5.9,64120,114 min,tt21191806,114.0
2,An Action Hero,2022,"Action, Comedy, Crime",Anirudh Iyer,"Ayushmann Khurrana, Jaideep Ahlawat, Gautam Jo...",7.0,32711,130 min,tt15600222,130.0
3,Missing in Action,1984,"Action, Adventure, Drama",Joseph Zito,"Chuck Norris, M. Emmet Walsh, David Tress",5.5,18026,101 min,tt0087727,101.0
4,Action Jackson,1988,"Action, Comedy, Crime",Craig R. Baxley,"Carl Weathers, Craig T. Nelson, Vanity",5.6,13166,96 min,tt0094612,96.0


In [ ]:
collector = MovieDataCollector(OMDB_API_KEY)

# Genre → search keywords (important workaround)
genre_queries = {
    "action": ["action"],
    "comedy": ["comedy"],
    "drama": ["drama"],
    "horror": ["horror"],
    "sci-fi": ["sci-fi"]
}

MOVIES_PER_GENRE = 10

all_movies = []

# -----------------------------
# SEARCH + FETCH
# -----------------------------
for genre, queries in genre_queries.items():
    print(f"Collecting {genre} movies...")
    collected = []

    for q in queries:
        if len(collected) >= MOVIES_PER_GENRE:
            break

        search_results = collector.search_movies(q, max_results=10)

        for m in search_results:
            if len(collected) >= MOVIES_PER_GENRE:
                break

            movie = collector.fetch_movie_by_id(m["imdbID"])
            if movie and genre.lower() in movie.get("Genre", "").lower():
                collected.append(movie)
                time.sleep(RATE_LIMIT_DELAY)

    all_movies.extend(collected)

# -----------------------------
# DEDUPLICATE
# -----------------------------
unique_movies = {}
for m in all_movies:
    unique_movies[m["imdbID"]] = m

final_movies = list(unique_movies.values())

print(f"\nTotal unique movies collected: {len(final_movies)}")

# -----------------------------
# DATAFRAME + CSV
# -----------------------------
df = collector.to_dataframe(final_movies)
df.to_csv("genre_based_movies.csv", index=False)

df.head()



Total unique movies collected: 33


,title,year,genre,director,actors,imdb_rating,imdb_votes,runtime,imdb_id,runtime_min
0,Last Action Hero,1993,"Action, Adventure, Comedy",John McTiernan,"Arnold Schwarzenegger, F. Murray Abraham, Art ...",6.5,171378,130 min,tt0107362,130.0
1,Back in Action,2025,"Action, Comedy",Seth Gordon,"Jamie Foxx, Cameron Diaz, McKenna Roberts",5.9,64120,114 min,tt21191806,114.0
2,An Action Hero,2022,"Action, Comedy, Crime",Anirudh Iyer,"Ayushmann Khurrana, Jaideep Ahlawat, Gautam Jo...",7.0,32711,130 min,tt15600222,130.0
3,Missing in Action,1984,"Action, Adventure, Drama",Joseph Zito,"Chuck Norris, M. Emmet Walsh, David Tress",5.5,18026,101 min,tt0087727,101.0
4,Action Jackson,1988,"Action, Comedy, Crime",Craig R. Baxley,"Carl Weathers, Craig T. Nelson, Vanity",5.6,13166,96 min,tt0094612,96.0


### Question 6.4: Data Quality Analysis

Using the dataset you created:

1. How many movies have missing IMDB ratings?
2. How many movies have missing box office data?
3. What's the distribution of ratings? (min, max, mean, median)
4. Which directors appear most frequently?
5. What's the average runtime by genre?

These quality checks will be important for Week 2 (Data Validation)!

In [ ]:
# YOUR CODE HERE
missing_imdb = df["imdb_rating"].isna().sum()
print(missing_imdb)
# missing_box_office = df["box_office"].isna().sum()
# missing_box_office
rating_stats = df["imdb_rating"].describe()[["min", "max", "mean", "50%"]]
rating_stats.rename({"50%": "median"})
print(rating_stats)
top_directors = (
    df["director"]
    .dropna()
    .value_counts()
    .head(10)
)

print(top_directors)
df_genre = df.copy()
df_genre["genre"] = df_genre["genre"].str.split(", ")
df_genre = df_genre.explode("genre")
avg_runtime_by_genre = (
    df_genre
    .groupby("genre")["runtime_min"]
    .mean()
    .sort_values(ascending=False)
)

print(avg_runtime_by_genre)


0
min     2.500000
max     7.800000
mean    6.330303
50%     6.500000
Name: imdb_rating, dtype: float64
director
John McTiernan               1
Seth Gordon                  1
Anirudh Iyer                 1
Joseph Zito                  1
Craig R. Baxley              1
Lance Hool                   1
Martin Scorsese              1
Woody Allen                  1
Alain Berbérian              1
Stephen Chow, Lik-Chi Lee    1
Name: count, dtype: int64
genre
Action         111.833333
Crime          111.666667
Thriller       100.000000
Comedy          98.705882
Adventure       97.000000
Fantasy         96.500000
Musical         94.000000
Mystery         92.666667
Horror          92.615385
Drama           90.750000
Documentary     89.250000
Romance         88.333333
Sci-Fi          87.000000
Name: runtime_min, dtype: float64


---

# Part 7: Challenge Problems

These are optional advanced exercises for those who finish early.

### Challenge 7.1: Rate Limit Handler

Create a `RateLimiter` class that:
1. Tracks how many requests have been made
2. Automatically adds delays to stay under a rate limit
3. Handles 429 (Too Many Requests) responses by waiting and retrying

```python
limiter = RateLimiter(requests_per_minute=30)
response = limiter.get("https://api.example.com/data")
```

In [ ]:
# YOUR CODE HERE


### Challenge 7.2: Async Movie Collector

The synchronous approach is slow because we wait for each request to complete.

Create an async version using `aiohttp` that can fetch multiple movies concurrently (while still respecting rate limits).

Compare the time to fetch 20 movies with sync vs async approach.

In [ ]:
# YOUR CODE HERE
# Hint: You'll need to install aiohttp: pip install aiohttp
# And use asyncio to run the async code


### Challenge 7.3: Multi-Source Data Fusion

Create a data collection pipeline that:
1. Fetches basic movie data from OMDb
2. Enriches it with additional data from another source (e.g., Wikipedia API for plot summaries)
3. Merges the data based on movie title/year
4. Handles cases where data is missing from one source

Wikipedia API example:
```
https://en.wikipedia.org/api/rest_v1/page/summary/Inception_(film)
```

In [ ]:
# YOUR CODE HERE


---

# Summary

In this lab, you learned:

1. **HTTP Fundamentals**: URLs, status codes, headers
2. **curl**: Command-line HTTP requests
3. **Python requests**: Programmatic data collection
4. **Error handling**: Timeouts, retries, status codes
5. **OMDb API**: Real-world movie data
6. **BeautifulSoup**: Web scraping when APIs don't exist
7. **Data pipelines**: Building reusable collection code

## Next Week

**Week 2: Data Validation & Quality**

The data we collected today is messy! Next week we'll learn:
- Schema validation with Pydantic
- Data type cleaning
- Handling missing values
- Quality metrics

---

## Submission

Save your completed notebook and submit:
1. This notebook with all cells executed
2. The CSV file of movies you collected
3. A brief summary (1 paragraph) of what you learned